# Intelligent Stock Market Trend Predictor using ML
**Step-by-step notebook for viva explanation** - BTech CSE / AIML Final Year Project

> Educational purpose only. Not financial advice.

Pipeline: Data Collection -> Preprocessing -> Feature Engineering -> Model Training -> Evaluation -> Deployment

Run the cells from top to bottom. Keep this notebook in the same folder as `stock_utils.py` and `visualization.py`.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

import stock_utils as su        # Modules 1, 2, 3
import visualization as viz     # Module 4

TICKER = "TCS.NS"               # change to RELIANCE.NS or INFY.NS to try others

## Step 1 - Data Collection (Module 1)
We download 2 years of daily OHLCV data (Open, High, Low, Close, Volume) from Yahoo Finance using `yfinance`.
Missing values are **forward-filled** only (never back-filled, because back-fill copies the future into the past).

In [ ]:
df = su.fetch_stock_data(TICKER, "2y")
print("Rows:", len(df), "| From", df.index[0].date(), "to", df.index[-1].date())
print("Missing values per column:")
print(df.isna().sum())
df.tail()

## Step 2 - Feature Engineering (Module 2)
Technical indicators (all use only past prices):
- **MA20 / MA50** - average closing price of the last 20 / 50 days (trend direction)
- **RSI(14)** - momentum from 0 to 100; above 70 overbought, below 30 oversold
- **MACD** - 12-day EMA minus 26-day EMA, with a 9-day signal line
- **Bollinger Bands** - 20-day average plus/minus 2 standard deviations (volatility)
- **Lag features** - Close 1, 2 and 5 days ago

**Target:** `Trend = 1 if Close(t+1) > Close(t) else 0`

In [ ]:
df_ind = su.add_indicators(df)                  # for charts (keeps all rows)
table = su.build_feature_table(df)              # indicators + Target, NaN rows removed
print("Rows after removing the first ~50 days without MA50:", len(table))
table[["Close", "MA20", "MA50", "RSI", "MACD", "Close_lag1", "Target"]].tail()

In [ ]:
viz.candlestick_chart(df_ind, TICKER).show()
viz.rsi_chart(df_ind, TICKER).show()
viz.macd_chart(df_ind, TICKER).show()

### Check the target
The last row has no "tomorrow", so its Target is NaN. We keep it only for predicting tomorrow, and remove it for training.
Also check the class balance: if UP and DOWN are close to 50-50, then accuracy around 50% means "no better than a coin flip".

In [ ]:
data = table.dropna(subset=["Target"])
print(data["Target"].value_counts())
print((data["Target"].value_counts(normalize=True) * 100).round(1))

## Step 3 - Time-based split (80 : 20, NO shuffle)
First 80% of days = training (older), last 20% = testing (newer).

**Data leakage prevention (viva answer):**
1. Never shuffle time series - shuffling lets the model train on days that come *after* the test days.
2. The scaler is fitted on the training part only.
3. Indicators and lag features use only past data.
4. Missing values are forward-filled only.
5. The last row (unknown tomorrow) is removed from training.

In [ ]:
X = data[su.FEATURES].values
y = data["Target"].values.astype(int)

split = su.time_split_index(len(data))
print("Train rows:", split, "->", data.index[0].date(), "to", data.index[split - 1].date())
print("Test rows :", len(data) - split, "->", data.index[split].date(), "to", data.index[-1].date())

X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]

## Step 4A - Model A: RandomForest (Baseline)
**Why RandomForest as baseline?** Simple to explain (many decision trees voting), hard to overfit, works on small data,
needs no scaling, and gives feature importance. If LSTM cannot beat it, LSTM is not worth the extra complexity.

In [ ]:
rf = su.build_random_forest()
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_metrics, rf_cm = su.evaluate(y_test, rf_pred)
print("RandomForest:", rf_metrics)
print("Confusion matrix [[TN, FP], [FN, TP]]:", rf_cm)

In [ ]:
importance = dict(zip(su.FEATURES, rf.feature_importances_.round(4)))
viz.feature_importance_chart(importance).show()

## Step 4B - Model B: LSTM
**Why LSTM for time series?** Stock data is a sequence. A RandomForest sees one day at a time. An LSTM reads the last
60 days *in order* and keeps a memory (cell state).

**The three gates**
| Gate | Question it answers |
|---|---|
| Forget gate | What part of the old memory should I erase? |
| Input gate | What new information from today should I store? |
| Output gate | What part of the memory should I send out as today's output? |

New memory = (old memory x forget gate) + (new information x input gate).
This mostly-additive update avoids the vanishing gradient problem of a normal RNN.

**Architecture:** Input (60 days) -> LSTM(50) -> Dropout(0.2) -> LSTM(50) -> Dropout(0.2) -> Dense(1, sigmoid), Adam optimizer.

In [ ]:
# Scaler learns mean/std from TRAIN rows only (prevents leakage)
scaler = StandardScaler().fit(X_train)
scaled = scaler.transform(X)

# Convert to 60-day windows: shape = (samples, 60, number of features)
X_seq, y_seq, end_idx = su.make_sequences(scaled, y)
is_train = end_idx < split                  # window belongs to train if it ENDS before the split
X_tr, y_tr = X_seq[is_train], y_seq[is_train]
X_te, y_te = X_seq[~is_train], y_seq[~is_train]
print("Train windows:", X_tr.shape, "| Test windows:", X_te.shape)
assert np.array_equal(y_te, y_test)         # same test days as the RandomForest

In [ ]:
import tensorflow as tf
tf.random.set_seed(42)
from tensorflow.keras.callbacks import EarlyStopping

lstm = su.build_lstm(n_features=X.shape[1])
lstm.summary()

In [ ]:
stop_early = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
history = lstm.fit(X_tr, y_tr, epochs=50, batch_size=16,
                   validation_split=0.1, shuffle=False,      # keep time order
                   callbacks=[stop_early], verbose=0)
print("Epochs run:", len(history.history["loss"]))

plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title("LSTM training curve")
plt.show()

In [ ]:
lstm_prob = lstm.predict(X_te, verbose=0).ravel()
lstm_pred = (lstm_prob >= 0.5).astype(int)
lstm_metrics, lstm_cm = su.evaluate(y_test, lstm_pred)
print("LSTM:", lstm_metrics)
print("Confusion matrix [[TN, FP], [FN, TP]]:", lstm_cm)

## Step 5 - Evaluation and comparison
- **Accuracy** = correct predictions / all predictions
- **Precision** = of days predicted UP, how many were really UP
- **Recall** = of days that were really UP, how many we caught
- **F1** = balance of precision and recall
- **Confusion matrix** = shows *how* the model is wrong (e.g. always predicting UP)

Objective: accuracy above 55%. The test set is small (about 90 days), so results are noisy - report honestly.

In [ ]:
comparison = pd.DataFrame({"RandomForest": rf_metrics, "LSTM": lstm_metrics}).T
display(comparison)
viz.model_comparison_bar({"RandomForest": rf_metrics, "LSTM": lstm_metrics}).show()

In [ ]:
fig_a = viz.confusion_matrix_figure(rf_cm, "RandomForest")
fig_b = viz.confusion_matrix_figure(lstm_cm, "LSTM")
plt.show()

In [ ]:
dates = [d.strftime("%Y-%m-%d") for d in data.index[split:]]
viz.actual_vs_predicted_chart(dates, y_test, {"RandomForest": rf_pred, "LSTM": lstm_pred}).show()

## Step 6 - Save models and predict tomorrow
The web app loads `.pkl` (RandomForest, scaler) and `.h5` (LSTM) files from the `models/` folder.
`train_and_save()` repeats the steps above in one call and also stores the results used by the app's Model Performance tab.

In [ ]:
results = su.train_and_save(TICKER, "2y")
print("Saved files in models/ for", TICKER)
print("RandomForest:", results["rf"]["metrics"])
print("LSTM        :", results["lstm"]["metrics"] if results["lstm"] else "skipped")

In [ ]:
out = su.predict_tomorrow(TICKER)
for name, p in [("RandomForest", out["rf_prob_up"]), ("LSTM", out["lstm_prob_up"])]:
    if p is not None:
        trend, conf = su.to_trend(p)
        print(f"{TICKER} - Tomorrow ({name}): {trend} Trend (Confidence {conf}%)")
print("Educational only - not financial advice.")

## Viva quick revision
1. **Why trend instead of price?** Direction is simpler and less noisy; exact price errors are hard to interpret.
2. **Why shuffle=False / time split?** To avoid data leakage; the model must never see the future.
3. **Why RandomForest baseline?** Simple, explainable, robust on small data, gives feature importance.
4. **Why LSTM?** Remembers order across 60 days using gates.
5. **Why Dropout 0.2?** Randomly drops 20% neurons during training to reduce overfitting.
6. **Why sigmoid output?** Gives probability of UP between 0 and 1 (used as confidence).
7. **Why is accuracy near 50-55%?** Markets are almost random day to day and news is not in our features.
8. **Future scope:** news sentiment analysis, more stocks, walk-forward validation.